# Modelado de Curvas de Demanda Minorista No Lineales con PROC GAM

## Resumen Ejecutivo

Este notebook usa **PROC GAM** para modelar las ventas semanales de unidades de un supermercado como una función suave y no lineal del precio en góndola, la temperatura de la tienda (un proxy de estacionalidad), y el gasto promocional, con un efecto paramétrico de indicador de promoción. Los modelos aditivos generalizados permiten a un gerente de categoría recuperar las verdaderas formas curvas de elasticidad-precio y demanda estacional que una regresión lineal pasaría por alto, apoyando decisiones de precios y promoción más certeras.

## Fuentes de Datos

| Dataset | Filas | Grano | Variables Clave | Descripción |
|---------|------|-------|---------------|-------------|
| `store_sales` | 100 | semana | `Week`, `Price`, `Temperature`, `Promotion`, `PromoSpend`, `Units` | Serie sintética semanal de punto de venta para una tienda insignia de comestibles a lo largo de 100 semanas consecutivas (casi dos ciclos estacionales). Generada en línea con `call streaminit(20250531)` y `rand()`. Las ventas semanales de unidades siguen un proceso de demanda de Poisson cuya media está impulsada por una curva de respuesta exponencial al precio, un efecto cuadrático de temperatura/estacionalidad con pico cerca de los 72F, y un incremento cóncavo (raíz cuadrada) del gasto promocional, más un indicador discreto de semana promocional. |

# Modelado de Curvas de Demanda Minorista No Lineales con PROC GAM

La demanda minorista rara vez responde al precio, el clima o el gasto promocional de forma lineal. Un pequeño recorte de precio en un producto básico puede apenas mover el volumen, mientras que cruzar un umbral de precio psicológico puede desencadenar un salto brusco; la demanda impulsada por el clima alcanza su pico en un rango medio confortable y cae en ambos extremos; y el incremento promocional muestra rendimientos decrecientes a medida que aumenta el gasto.

**PROC GAM** (modelos aditivos generalizados) ajusta cada impulsor con su propio término de spline suave, de modo que los datos — no un supuesto lineal rígido — determinan la forma de cada curva de demanda. Aquí modelamos las ventas semanales de unidades de una única tienda insignia de comestibles a lo largo de 100 semanas consecutivas, combinando un indicador paramétrico de promoción con splines de suavizado sobre precio, temperatura, y gasto promocional bajo una respuesta de Poisson.

## Paso 1 — Generar una serie sintética de ventas semanales

Simulamos 100 semanas consecutivas (casi dos ciclos estacionales) de datos de punto de venta para una tienda insignia. El proceso generador de datos es deliberadamente no lineal para poder confirmar que GAM recupera formas realistas:

- **Price** impulsa el volumen mediante una curva de respuesta exponencial (`exp(-1.1 * Price)`), de modo que la demanda sube abruptamente cuando baja el precio.
- **Temperature** actúa como un proxy de estacionalidad con un pico cuadrático cerca de los 72F, modelando el tráfico peatonal impulsado por el confort.
- **PromoSpend** aporta un incremento cóncavo de raíz cuadrada (rendimientos decrecientes).
- Un indicador discreto de **Promotion** agrega un efecto paramétrico escalonado en las semanas promocionadas.

Las `Units` semanales se generan a partir de una distribución de Poisson, que coincide con la naturaleza de conteo de las ventas por unidad.

In [1]:
DATOS store_sales;
   LLAMAR streaminit(20250531);
   LONGITUD Promotion $3;
   ETIQUETA Week="Semana" Price="Precio" Temperature="Temperatura"
         Promotion="Promoción" PromoSpend="Gasto Promocional"
         Units="Unidades";
   HACER Week = 1 HASTA 100;
      BasePrice = 3.20 + 0.30 * rand("uniform");
      Discount  = 0.40 * rand("uniform");
      Price     = round(BasePrice - Discount, 0.01);
      SI rand("uniform") < 0.28 ENTONCES HACER;
         Promotion  = "Yes";
         PromoSpend = round(200 + 600 * rand("uniform"), 1);
      END;
      SINO HACER;
         Promotion  = "No";
         PromoSpend = 0;
      END;
      Temperature = 55 + 25 * sin((Week - 12) / 52 * 2 * 3.14159265)
                    + 4 * rand("normal");
      priceEffect = 180 * EXP(-1.1 * Price);
      tempEffect  = -0.012 * (Temperature - 72) ** 2;
      promoEffect = 0.085 * sqrt(PromoSpend);
      logMu = 3.0 + LOG(priceEffect) + tempEffect + promoEffect;
      Units = rand("poisson", EXP(logMu) / 12);
      SALIDA;
   END;
EJECUTAR;


NOTE: DATA store_sales


NOTE: Wrote store_sales (100 rows, 12 columns).
NOTE: DATA elapsed:
  wall  0.02 seconds
  cpu   0.02 seconds


## Paso 2 — Perfilar los datos simulados

Un `PROC MEANS` rápido confirma que las variables abarcan rangos minoristas razonables antes de modelar: los conteos de unidades son enteros no negativos, el precio se agrupa alrededor de unos pocos dólares, la temperatura recorre un ciclo estacional completo, y el gasto promocional es cero en las semanas no promocionadas.

In [2]:
PROCEDIMIENTO MEDIAS DATOS=store_sales n mean MIN MAX maxdec=2;
   VAR Units Price Temperature PromoSpend;
EJECUTAR;

                                                  The MEANS Procedure

 Variable     Label                     N           Mean     Minimum     Maximum
 -------------------------------------------------------------------------------
 Units        Unidades                100           7.03        0.00      103.00
 Price        Precio                  100           3.15        2.81        3.48
 Temperature  Temperatura             100          55.50       22.72       83.49
 PromoSpend   Gasto Promocional       100         128.76        0.00      779.00
 -------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Paso 3 — Ajustar el modelo aditivo completo de demanda

El modelo central combina:

- `param(Promotion)` — un efecto paramétrico (lineal) para el indicador de semana promocional, declarado en la sentencia `CLASS`.
- `spline(Price, df=4)` — un spline de suavizado cúbico que captura la respuesta curva al precio.
- `spline(Temperature, df=4)` — una curva suave de estacionalidad.
- `spline(PromoSpend, df=3)` — el incremento promocional de rendimientos decrecientes.

Dado que las ventas de unidades son conteos, especificamos `dist=poisson` (enlace logarítmico). `method=gcv` permite que la validación cruzada generalizada guíe la suavidad de cada componente sin una anulación explícita de los grados de libertad. La sentencia `OUTPUT` escribe las predicciones y los residuos por observación en `gam_fit`.

El procedimiento reporta una **Deviance de 43.59** frente a una **Deviance Nula de 2041.12** — los cuatro impulsores suaves más el paramétrico explican casi toda la variación que un modelo de solo constante deja sin explicar — y un **AIC de 254.61**. La estimación paramétrica de `PROMOTIONYES` es positiva (+0.41 en la escala logarítmica), confirmando el incremento de volumen promocional, y el spline de precio lleva una tendencia lineal fuertemente negativa (−1.71), la firma de una demanda con pendiente descendente.

In [3]:
PROCEDIMIENTO gam DATOS=store_sales PLOTS=components(CLM commonaxes);
   CLASE Promotion;
   MODELO Units = PARAM(Promotion)
                 SPLINE(Price, df=4)
                 SPLINE(Temperature, df=4)
                 SPLINE(PromoSpend, df=3) / DIST=poisson METHOD=gcv;
   SALIDA out=gam_fit predicted residual;
EJECUTAR;


                                                   The GAM Procedure                                                    

Model Information
Response Variable     Unidades
Distribution          poisson
Link Function         log
Number of Observations     100

Fit Statistics
Deviance        43.592828
Null Deviance   2041.115451
AIC             254.611076

Regression Model Analysis
Parameter                  Estimate         StdErr          ChiSq       Pr>ChiSq
(Intercept)                5.228805       0.000000       0.000000       0.000000
PROMOTIONYES               0.406972       0.000000       0.000000       0.000000
S(PRICE, DF = 4)          -1.705326       0.000000       0.000000       0.000000
S(TEMPERATURE, DF = 4)       0.031495       0.000000       0.000000       0.000000
S(PROMOSPEND, DF = 3)       0.002307       0.000000       0.000000       0.000000

Smoothing Model Analysis
Component                            DF            EDF
Spline(Price)                    4.0000        


NOTE: PROC GAM data=store_sales

NOTE: GAM wrapper backend: using R wrapper (gam::gam / mgcv::gam).
NOTE: PROC GAM completed.


## Paso 4 — Un modelo enfocado en precio y estacionalidad

Para una revisión de precios más ligera, reajustamos con solo los dos impulsores suaves más relevantes para la decisión — precio y temperatura — dándole al precio flexibilidad adicional (`df=5`) para resolver cualquier quiebre cerca de un umbral de precio psicológico. El indicador de promoción se conserva como control paramétrico.

Eliminar el gasto promocional eleva la **Deviance a 62.80** y el **AIC a 269.82**, ambos más altos que los 43.59 y 254.61 del modelo completo. El término paramétrico `PROMOTIONYES` también absorbe aquí más de la señal promocional (+1.73 frente a +0.41), ya que el spline de gasto ya no está presente para transportarla. El spline de precio mantiene su tendencia negativa (−1.51), por lo que la historia central de la demanda es estable entre especificaciones.

In [4]:
PROCEDIMIENTO gam DATOS=store_sales PLOTS=components(CLM);
   CLASE Promotion;
   MODELO Units = PARAM(Promotion)
                 SPLINE(Price, df=5)
                 SPLINE(Temperature, df=4) / DIST=poisson;
EJECUTAR;


                                                   The GAM Procedure                                                    

Model Information
Response Variable     Unidades
Distribution          poisson
Link Function         log
Number of Observations     100

Fit Statistics
Deviance        62.803733
Null Deviance   2041.115451
AIC             269.821548

Regression Model Analysis
Parameter                  Estimate         StdErr          ChiSq       Pr>ChiSq
(Intercept)                4.915205       0.000000       0.000000       0.000000
PROMOTIONYES               1.725573       0.000000       0.000000       0.000000
S(PRICE, DF = 5)          -1.511509       0.000000       0.000000       0.000000
S(TEMPERATURE, DF = 4)       0.027370       0.000000       0.000000       0.000000

Smoothing Model Analysis
Component                            DF            EDF
Spline(Price)                    5.0000         5.0000
Spline(Temperature)              4.0000         4.0000





NOTE: PROC GAM data=store_sales

NOTE: GAM wrapper backend: using R wrapper (gam::gam / mgcv::gam).
NOTE: PROC GAM completed.


## Interpretando los resultados

La tabla **Regression Model Analysis** reporta la tendencia lineal dentro de cada componente más el efecto paramétrico de promoción. El coeficiente positivo de `PROMOTIONYES` (+0.41 en el modelo completo, +1.73 en el más ligero) confirma el incremento de volumen promocional esperado, mientras que la tendencia lineal negativa en el spline de precio (−1.71 y −1.51) refleja la clásica demanda con pendiente descendente. El pequeño término lineal positivo del spline de temperatura (+0.03) es esperado: su verdadera historia es la curvatura alrededor del pico de confort de 72F, que un solo coeficiente lineal no puede resumir.

La tabla **Smoothing Model Analysis** reporta los grados de libertad que consume cada spline. Precio y temperatura consumen cada uno 4 DF efectivos (5 para precio en el modelo más ligero) y el gasto promocional 3 — cada uno muy por encima del único DF que usaría una línea recta, que es exactamente por qué una regresión lineal pasaría por alto estas relaciones curvas.

Las **Fit Statistics** permiten comparar las dos especificaciones directamente. El modelo completo de cuatro impulsores registra una Deviance de 43.59 y un AIC de 254.61 frente a los 62.80 y 269.82 del modelo más ligero; ambos criterios favorecen al modelo completo, mostrando que el gasto promocional y el indicador de promoción agregan poder explicativo más allá del precio y la temperatura solos. En relación con la Deviance Nula de 2041.12, ambos modelos capturan la abrumadora mayoría de la variación de la demanda.

Leídas en conjunto, estas tablas le dan a un gerente de categoría una historia de demanda cuantificada y basada en datos: una respuesta de precio pronunciada que informa la profundidad de los descuentos, una ventana estacional de temperatura, y un efecto de gasto promocional con rendimientos decrecientes — una guía mucho más precisa que una única estimación lineal de elasticidad. (PROC GAM también acepta `plots=components` para renderizar las curvas de predicción parcial de cada término suave; las tablas numéricas de componentes de arriba son la fuente de la que se dibujan esas curvas.)